# Qari Reader · قارئ — Arabic OCR on a free GPU

This notebook runs **Qari Reader** (Arabic OCR built on the open-source Qari-OCR model) on Google Colab's free GPU.

**Before you start:** menu **Runtime → Change runtime type → T4 GPU → Save**.

Then run the cells from top to bottom with ▶.

## 1. Get the code and install it
This installs straight from GitHub. (To use a zip instead, set `REPO_URL = ""` and upload `qari-reader.zip` when asked.)

In [ ]:
REPO_URL = "https://github.com/fekovic000-ops/Qari-OCR"

import os, subprocess, zipfile
os.chdir("/content")
if not os.path.exists("/content/qari-reader"):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/qari-reader"], check=True)
    else:
        from google.colab import files
        up = files.upload()                       # choose qari-reader.zip
        name = next(iter(up))
        zipfile.ZipFile(name).extractall("/content")
os.chdir("/content/qari-reader")
!pip -q install -e ".[web,gpu-lowmem]"
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch the runtime to T4 GPU!")

## 2a. Easiest: open the web app
Run this cell and click the **public URL** it prints (ending in `gradio.live`). Upload your PDF there, press **Start**, and download the .txt / .docx when it finishes. The link works on your phone too while this notebook is running.

In [ ]:
!python -m qari_reader.app --share

## 2b. Or: run a whole book from here
Upload one or more PDFs, then the results are zipped and downloaded. If Colab disconnects, run the cell again: finished pages are kept (save `output/` to Google Drive if you want them to survive a full reset).

In [ ]:
from google.colab import files
uploaded = list(files.upload().keys())

PAGES   = ""        # e.g. "1-50"; empty = all pages
OPTIONS = ""        # e.g. "--enhance --columns" ; "--model v0.3" for handwriting / layout

args = " ".join(f'"{u}"' for u in uploaded)
pages = f'--pages "{PAGES}"' if PAGES else ""
!python -m qari_reader {args} {pages} {OPTIONS} --format txt,docx,md --batch 2 -o output

!cd output && zip -qr ../results.zip . && cd ..
files.download("results.zip")

### Tips
- Faded or yellowed scans: add `--enhance`.
- Two-column pages: add `--columns`.
- Very small print: add `--max-pixels 2500000`.
- Out of GPU memory: use `--batch 1` or `--quantize 8bit`.
- Pages the tool wasn't sure about are listed at the end — check those by eye.